# This notebook is a test play ground of how I will scrape the data and format to build the scraping script


In [1]:
from  bs4 import BeautifulSoup
import requests
import pandas as pd
import time 
import yaml


In [103]:
with open('../config.yaml', 'r') as f:
    config = yaml.safe_load(f)

In [ ]:
# First we are going to construct our dataframe to hold our data evantullay 
# After some research I found how the data I will get will be structure
# The data will have the following fields: 
fields = ['title', 'property_type', 'sale/rent', 'city','district','area', 'neighborhood', 'property_subtype', 'ownership type','electricy meter','water meter', 'covered parking', 'natural gas', 'security', 'pets allowed','balcony', 'private garden', 'pool', 'landline','maid room','built in kitchen','elevator','central A/C & heating', 'furnished', 'floor_level', 'rooms', 'bathrooms', 'payment option','delivery time','price', 'seller_name', 'area_size', 'listing_date']
df = pd.DataFrame(columns=fields)

In [4]:


print('Number of fields to extract:', len(fields))

Number of fields to extract: 32


In [11]:
# Here is a we will try to scrape a single listing page to test our selectors 
# So there are some information that we already know about the listing 
# 1. property type: apartment 2. sale/rent: sale 3. city: cairo 4. district: new cairo 5. area: fifth settlement 6. neighborhood: mountain view i city compound
# The third decision all urls will be typed in the config.yaml file which will not be shared publicly
url = config['data_source']['realstate_website']['test_listing_url']
headers = config['data_source']['realstate_website']['headers']
response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.text, 'lxml')

In [117]:
# for now we will just get the data and put in a dictonary then we will make the appending logic later
page_data = {}
page_title = soup.select_one(config['data_source']['realstate_website']['selectorrs']['title']).text.strip()
page_data['title'] = page_title
page_data['price'] = soup.select_one(config['data_source']['realstate_website']['selectorrs']['price']).text.strip()
page_data['creation_date'] = soup.select_one(config['data_source']['realstate_website']['selectorrs']['creation_date']).text.strip()
page_data['seller_name'] = soup.select_one(config['data_source']['realstate_website']['selectorrs']['seller_name']).text.strip()
for key, value in page_data.items():
    print(f"{key}: {value}")


title: Apartment for sale 160M prime location lowest price in mountain view i city
price: EGP 6,000,000
creation_date: 5 days ago
seller_name: Views Real Estate


In [118]:
# Here comes the tricky part of extracting the data for the highlihts, details and features sections
# First we will extract the highlights section
def highlights_extractor(highlit_container, page_data):
    #   outer items contains 
    outer_items = highlit_container.select(config['data_source']['realstate_website']['selectorrs']['higlight_items'])

    for item in outer_items:
        spans = item.find_all('span')
        if len(spans) >=2:
            key = spans[0].text.strip().lower()
            value = spans[1].text.strip()
            page_data[key] = value
            print(f"Highlight - {key}: {value}")
    return page_data

In [119]:
highlights_container = soup.select_one(config['data_source']['realstate_website']['selectorrs']['highlights_container'])
page_data = highlights_extractor(highlights_container, page_data)


Highlight - type: Apartment
Highlight - ownership: Resale
Highlight - area (m²): 160
Highlight - bedrooms: 3
Highlight - bathrooms: 3
Highlight - furnished: No


In [120]:
details_container = soup.select_one('div[aria-label="Details"]')
# print(details_container.prettify())
details_itemss = details_container.select(config['data_source']['realstate_website']['selectorrs']['details_items'])
for item in details_itemss:
    spans = item.find_all('span')
    key = spans[0].text.strip().lower()
    value = spans[1].text.strip().lower()
    page_data[key] = value
    print(f"Detail - {key}: {value}")

Detail - level: 1
Detail - payment option: cash
Detail - completion status: ready


In [122]:
features_container = soup.select_one(config['data_source']['realstate_website']['selectorrs']['features_container'])
features_items = features_container.select(config['data_source']['realstate_website']['selectorrs']['features_items'])


In [123]:
for item in features_items:
    spans = item.find_all('span')
    for span in spans:
        key = span.text.strip().lower()
        if key == 'see all':
            continue
        
        page_data[key] = True
        print(f"Feature - {key}: True")

Feature - balcony: True
Feature - built in kitchen appliances: True
Feature - security: True
Feature - covered parking: True
Feature - pets allowed: True
Feature - pool: True
Feature - electricity meter: True
Feature - elevator: True


In [126]:
for key, value in page_data.items():
    print(f'{key} : {value}')

title : Apartment for sale 160M prime location lowest price in mountain view i city
price : EGP 6,000,000
creation_date : 5 days ago
seller_name : Views Real Estate
type : Apartment
ownership : Resale
area (m²) : 160
bedrooms : 3
bathrooms : 3
furnished : No
level : 1
payment option : cash
completion status : ready
balcony : True
built in kitchen appliances : True
security : True
covered parking : True
pets allowed : True
pool : True
electricity meter : True
elevator : True


##### Now we want to know how to get the data from each page so we inspect the page and start taking notes
##### then I will try to take one property and see how to get its data then I will from there build the logic 
##### but we need first to understand that each filter will change the url and the data we get
##### to get all data we need to loop over all filters and get the data from each page and be aware of dublicates but this problem will be solved later 
##### let's now try to get the data from one property 
#### for example this one will be in: 
1. Cairo as all the data we will be collecting
2. Apartment sale
3. In New Cairo
4. 5th setlment
5. mountain-view-icity-compound<br>
**Lastly All the data will be taken from the property page not like in the generlization some will be taken before entering the property page**

In [ ]:
# this is a trail to understand what the request get 
url = "https://www.dubizzle.com.eg/en/ad/apartment-for-sale-160m-prime-location-lowest-price-in-mountain-view-i-city-ID503018452.html/"
# headers is to mimic a real browser request so the server doesn't block us
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
}
response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.text, 'lxml')

In [8]:
# The districts names are under a div with class "d36fe94e"
# each listing is is under a <a> tag containing a hyperlink to filter the properties with this district
# then under the a tag there is a span tag containing the district name and other span tag containing the count of properties in this district
districts_div = soup.find('div', class_='d36fe94e')
districts = districts_div.find_all('a')
districts_list = []
for district in districts:
    if len(district.findAll('span')) < 2:
        continue
    link = district['href']
    name = district.find_all('span')[0].text
    count = district.find_all('span')[1].text
    districts_list.append({'link': link, 'name': name, 'count': count})

for d in districts_list:
    print(d)
       

{'link': '/en/properties/apartments-duplex-for-sale/new-cairo/', 'name': 'New Cairo', 'count': '(19908)'}
{'link': '/en/properties/apartments-duplex-for-sale/madinaty/', 'name': 'Madinaty', 'count': '(4483)'}
{'link': '/en/properties/apartments-duplex-for-sale/mostakbal-city/', 'name': 'Mostakbal City', 'count': '(3333)'}
{'link': '/en/properties/apartments-duplex-for-sale/new-capital-city/', 'name': 'New Capital City', 'count': '(2591)'}
{'link': '/en/properties/apartments-duplex-for-sale/shorouk-city/', 'name': 'Shorouk City', 'count': '(1543)'}
{'link': '/en/properties/apartments-duplex-for-sale/katameya/', 'name': 'Katameya', 'count': '(829)'}
{'link': '/en/properties/apartments-duplex-for-sale/sheraton/', 'name': 'Sheraton', 'count': '(819)'}
{'link': '/en/properties/apartments-duplex-for-sale/nasr-city/', 'name': 'Nasr City', 'count': '(816)'}
{'link': '/en/properties/apartments-duplex-for-sale/obour-city/', 'name': 'Obour City', 'count': '(729)'}
{'link': '/en/properties/apartme

C:\Users\marwa\AppData\Local\Temp\ipykernel_14480\991778122.py:8: DeprecationWarning: Call to deprecated method findAll. (Replaced by find_all) -- Deprecated since version 4.0.0.
  if len(district.findAll('span')) < 2:


40

In [9]:
try:
    import lxml
    print("✅ lxml imported successfully!")
    print("Version:", lxml.__version__)
except ImportError as e:
    print("❌ Error:", e)

✅ lxml imported successfully!
Version: 6.0.2


In [12]:
import subprocess

# Use uv to install instead of pip
result = subprocess.run(
    ['uv', 'pip', 'install', '--reinstall', 'lxml'],
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)


Using Python 3.9.2 environment at: D:\Marwan\Projects\cairo_real_state_intelligence_platform\.venv
Resolved 1 package in 651ms
Prepared 1 package in 0.83ms
Uninstalled 1 package in 27ms
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 1 package in 93ms
 ~ lxml==6.0.2

